# 🚀 VoxCPM2 - Khmer Zero-Shot Voice Cloning API Server
### Powered by OpenBMB VoxCPM2 (2B Parameters, 30 Languages, 48kHz Hi-Fi)

> **ជំហានដំបូង:** ចូល **Runtime -> Change runtime type -> ជ្រើសរើស T4 GPU -> Save** មុនចុច Run!

In [ ]:
# 1. ពិនិត្យ GPU
!nvidia-smi

In [ ]:
# 2. Clone និងដំឡើង VoxCPM2 + Dependencies ទាំងអស់
!git clone https://github.com/OpenBMB/VoxCPM.git /content/VoxCPM
%cd /content/VoxCPM
!pip install -q fastapi uvicorn python-multipart pyngrok soundfile numpy nest_asyncio
!pip install -q einops addict wetext modelscope funasr argbind torchcodec
!pip install -q -e /content/VoxCPM

In [ ]:
# 3. ចាប់ផ្តើម VoxCPM2 API Server & បង្កើត Public URL
import os
import sys

# បញ្ចូល Path ដើម្បីឱ្យរកឃើញម៉ូឌុល voxcpm 100%
sys.path.insert(0, "/content/VoxCPM/src")
sys.path.insert(0, "/content/VoxCPM")

import torch
import soundfile as sf
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok
import uvicorn
import nest_asyncio
import voxcpm

nest_asyncio.apply()

print("📥 Loading VoxCPM2 Model (2B weights from HuggingFace)...")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = voxcpm.VoxCPM.from_pretrained("openbmb/VoxCPM2", device=device, optimize=True)
print(f"✅ VoxCPM2 Model Ready on {device}!")

app = FastAPI(title="VoxCPM2 Khmer Voice API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

os.makedirs("/content/uploads", exist_ok=True)
os.makedirs("/content/outputs", exist_ok=True)

@app.post("/api/clone-and-speak")
async def clone_and_speak(
    text: str = Form(...),
    reference_audio: UploadFile = File(None),
    timesteps: int = Form(10),
    cfg_value: float = Form(2.0)
):
    try:
        ref_path = None
        if reference_audio and reference_audio.filename:
            ref_path = os.path.join("/content/uploads", reference_audio.filename)
            with open(ref_path, "wb") as f:
                f.write(await reference_audio.read())

        gen_kwargs = {
            "text": text.strip(),
            "cfg_value": cfg_value,
            "inference_timesteps": timesteps,
            "normalize": True,
            "denoise": True
        }
        if ref_path:
            gen_kwargs["reference_wav_path"] = ref_path

        wav = model.generate(**gen_kwargs)
        out_file = os.path.join("/content/outputs", f"out_{torch.randint(1000, 9999, (1,)).item()}.wav")
        sf.write(out_file, wav, 48000)
        return FileResponse(out_file, media_type="audio/wav")
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

tunnel = ngrok.connect(8000)
print("=" * 65)
print(f"🎉 VOXCPM2 API PUBLIC URL: {tunnel.public_url}")
print("👉 Copy URL នេះយកទៅដាក់ក្នុង .env នៅលើកុំព្យូទ័ររបស់អ្នក!")
print("=" * 65)

uvicorn.run(app, host="0.0.0.0", port=8000)